# Week 6: Single-Cell RNA-Seq Analysis

This notebook prepares the environment and data for a simplified scRNA-seq pipeline following Single-cell Best Practices (Section 3.8.2). It will:
- Build a cell-gene expression matrix (AnnData)
- Cluster cells with Leiden
- Annotate cell types with CellTypist

Deliverables:
- AnnData cell×gene matrix
- UMAP/Leiden clustering plot
- CellTypist annotation plot

Notes:
- Self-contained: installs dependencies, fetches data at runtime
- Python 3.13, no conda or pyroe
- Reference: https://www.sc-best-practices.org/introduction/raw_data_processing.html#a-real-world-example


## System Requirements Check

This section checks system and Python dependencies:
- System: `cmake`, `libtbb12`, `wget`, `curl`
- Python: `scanpy`, `anndata<0.11`, `leidenalg`, `celltypist`

Notes:
- No automatic `sudo apt-get` will be executed. Missing system packages will show a suggested command for you to run manually in a terminal.
- Python packages missing will be installed with `pip --user`.


In [ ]:
%%bash
set -u
export PATH="$HOME/.local/bin:$HOME/.cargo/bin:$PATH"

printf "\n=== System & Python Environment Check ===\n"

# --- System package checks (cmake, libtbb12, wget, curl) ---
missing_pkgs=()

check_cmd() {
  if command -v "$1" >/dev/null 2>&1; then
    printf "[OK] %s is installed at %s\n" "$1" "$(command -v "$1")"
  else
    printf "[MISSING] %s is not installed\n" "$1"
    missing_pkgs+=("$2")
  fi
}

# cmake, wget, curl are commands; libtbb12 is a Debian package
check_cmd cmake cmake
check_cmd wget wget
check_cmd curl curl

# Check libtbb12 via dpkg; fall back to checking shared object presence
if dpkg -s libtbb12 >/dev/null 2>&1; then
  echo "[OK] libtbb12 package is installed (dpkg)"
else
  if ls /usr/lib/*/libtbb.so.12 >/dev/null 2>&1; then
    echo "[OK] libtbb12 library found"
  else
    echo "[MISSING] libtbb12 (Intel TBB runtime)"
    missing_pkgs+=("libtbb12")
  fi
fi

if [ ${#missing_pkgs[@]} -gt 0 ]; then
  echo "\nMissing system packages detected: ${missing_pkgs[*]}"
  echo "Suggested install (requires sudo):"
  echo "  sudo apt-get update && sudo apt-get install -y ${missing_pkgs[*]}"
  echo "Note: This notebook will not run sudo automatically. Please run the command above in your terminal."
else
  echo "All required system packages are present."
fi

# --- Python package checks and installs (user-level) ---
python3 - <<'PY'
import sys, subprocess

print("\n--- Python packages ---")

def ensure(pkg, install_name=None, constraint=None):
    name = install_name or pkg
    spec = name if constraint is None else f"{name}{constraint}"
    try:
        mod = __import__(pkg)
        if pkg == 'anndata':
            ver = getattr(mod, '__version__', '0')
            def parse(v):
                parts = []
                for p in v.split('.'):
                    num = ''
                    for ch in p:
                        if ch.isdigit(): num += ch
                        else: break
                    parts.append(int(num or '0'))
                return tuple(parts + [0]*(3-len(parts)))
            if parse(ver) >= (0,11,0):
                print(f"[UPGRADE/DOWNGRADE] anndata {ver} -> enforcing <0.11")
                subprocess.run([sys.executable, '-m', 'pip', 'install', '--user', 'anndata<0.11'], check=False)
            else:
                print(f"[OK] anndata {ver}")
        else:
            print(f"[OK] {pkg} present")
    except Exception:
        print(f"[INSTALL] {spec}")
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--user', spec], check=False)

ensure('scanpy')
ensure('anndata', 'anndata', '<0.11')
ensure('leidenalg')
ensure('celltypist')

# Show pip location/version for clarity
subprocess.run([sys.executable, '-m', 'pip', '--version'])
PY

printf "\nEnvironment check complete. If system packages were missing, copy the suggested sudo apt-get command to your terminal and re-run this cell.\n"


## Data Download & Setup


In [ ]:
%%bash
set -u
export PATH="$HOME/.local/bin:$HOME/.cargo/bin:$PATH"

DATA_DIR="data"
BOX_URL="https://app.box.com/s/lx2xownlrhz3us8496tyu9c4dgade814"
TARBALL="toy_read_ref_set.tar.gz"
WL_GZ_URL="https://github.com/f0t1h/3M-february-2018/raw/refs/heads/master/3M-february-2018.txt.gz"
WL_GZ_PATH="$DATA_DIR/3M-february-2018.txt.gz"
WL_TXT_PATH="$DATA_DIR/3M-february-2018.txt"

mkdir -p "$DATA_DIR"

printf "\n=== Data Setup ===\n"

# 1) Always ensure whitelist first so subsequent messages aren't buried by progress
if [ -f "$WL_TXT_PATH" ]; then
  printf "Whitelist present: %s\n" "$WL_TXT_PATH"
else
  printf "Fetching 10x v3 whitelist ...\n"
  wget -q --show-progress -O "$WL_GZ_PATH" "$WL_GZ_URL" || { echo "Failed to download whitelist"; exit 1; }
  gunzip -f "$WL_GZ_PATH" || { echo "Failed to extract whitelist"; exit 1; }
  if [ -f "$WL_TXT_PATH" ]; then
    printf "Whitelist ready: %s\n" "$WL_TXT_PATH"
  else
    echo "Whitelist missing after extraction."; exit 1;
  fi
fi

# 2) Handle sample data presence / extraction
DATA_READY=0
# Canonical layout
if [ -d "$DATA_DIR/toy_ref_read/toy_read_fastq" ] && [ -d "$DATA_DIR/toy_ref_read/toy_human_ref" ]; then
  DATA_READY=1
fi

# If not canonical, normalize if top-level dirs exist
if [ "$DATA_READY" -eq 0 ] && [ -d "$DATA_DIR/toy_read_fastq" ] && [ -d "$DATA_DIR/toy_human_ref" ]; then
  printf "Found existing data at %s; normalizing layout to %s/toy_ref_read...\n" "$DATA_DIR" "$DATA_DIR"
  mkdir -p "$DATA_DIR/toy_ref_read"
  [ -d "$DATA_DIR/toy_ref_read/toy_read_fastq" ] || mv "$DATA_DIR/toy_read_fastq" "$DATA_DIR/toy_ref_read/" 2>/dev/null || true
  [ -d "$DATA_DIR/toy_ref_read/toy_human_ref" ] || mv "$DATA_DIR/toy_human_ref" "$DATA_DIR/toy_ref_read/" 2>/dev/null || true
  if [ -d "$DATA_DIR/toy_ref_read/toy_read_fastq" ] && [ -d "$DATA_DIR/toy_ref_read/toy_human_ref" ]; then
    DATA_READY=1
  fi
fi

# If still not ready, try to extract from tarball if present
if [ "$DATA_READY" -eq 0 ] && [ -f "$TARBALL" ]; then
  printf "Found %s. Extracting into %s/...\n" "$TARBALL" "$DATA_DIR"
  tar -xzf "$TARBALL" -C "$DATA_DIR" || { echo "Extraction failed"; exit 1; }
  # Normalize if tar created a top folder
  if [ -d "$DATA_DIR/toy_read_ref_set" ]; then
    [ -d "$DATA_DIR/toy_ref_read/toy_read_fastq" ] || mv "$DATA_DIR/toy_read_ref_set/toy_read_fastq" "$DATA_DIR/" 2>/dev/null || true
    [ -d "$DATA_DIR/toy_ref_read/toy_human_ref" ] || mv "$DATA_DIR/toy_read_ref_set/toy_human_ref" "$DATA_DIR/" 2>/dev/null || true
    rmdir "$DATA_DIR/toy_read_ref_set" 2>/dev/null || true
  fi
  # Ensure canonical layout under toy_ref_read
  mkdir -p "$DATA_DIR/toy_ref_read"
  [ -d "$DATA_DIR/toy_ref_read/toy_read_fastq" ] || mv "$DATA_DIR/toy_read_fastq" "$DATA_DIR/toy_ref_read/" 2>/dev/null || true
  [ -d "$DATA_DIR/toy_ref_read/toy_human_ref" ] || mv "$DATA_DIR/toy_human_ref" "$DATA_DIR/toy_ref_read/" 2>/dev/null || true

  if [ -d "$DATA_DIR/toy_ref_read/toy_read_fastq" ] && [ -d "$DATA_DIR/toy_ref_read/toy_human_ref" ]; then
    DATA_READY=1
    printf "OK: toy_ref_read directory prepared.\n"
    # Remove tarball after successful extraction to keep workspace clean
    rm -f "$TARBALL" && printf "Removed archive: %s\n" "$TARBALL" || true
  else
    echo "Warning: Expected folders not found after extraction. Contents of $DATA_DIR:"; ls -la "$DATA_DIR"
  fi
fi

# If data still missing, show Box instructions now (after whitelist progress) and exit gracefully
if [ "$DATA_READY" -eq 0 ]; then
  printf "[Action Required]\n"
  printf "Box requires a browser for download.\n"
  printf "1) Open: %s\n" "$BOX_URL"
  printf "2) Download file: %s\n" "$TARBALL"
  printf "3) Place it next to this notebook: %s/%s\n" "$(pwd)" "$TARBALL"
  printf "4) Re-run this cell.\n"
  exit 0
fi

# 3) Summary (only when data available)
printf "\nData directory summary:\n"
du -h --max-depth=2 "$DATA_DIR" 2>/dev/null || true

# List key files if present
if [ -d "$DATA_DIR/toy_ref_read/toy_human_ref/fasta" ]; then
  echo "Reference FASTA(s):"; ls -lh "$DATA_DIR/toy_ref_read/toy_human_ref/fasta" || true
fi
if [ -d "$DATA_DIR/toy_ref_read/toy_human_ref/genes" ]; then
  echo "GTF(s):"; ls -lh "$DATA_DIR/toy_ref_read/toy_human_ref/genes" || true
fi
if [ -d "$DATA_DIR/toy_ref_read/toy_read_fastq" ]; then
  echo "FASTQs:"; ls -lh "$DATA_DIR/toy_ref_read/toy_read_fastq" | head -n 20 || true
fi